# CV Preprocessing — INFORMATION-TECHNOLOGY

Pipeline: **Resume.csv + PDF folder → regex (HTML structure) → GPT-4.1 normalization → per-field embeddings → data points**

Each resume becomes **one data point per field** (`objective`, `work_exp`, `information`, `certification`, `technical_skills`):

```
point = {
    "id":      uuid5(resume_id + field),          # Qdrant-compatible
    "vector":  embedding(text of that field),     # text-embedding-3-small, 1536-d
    "payload": {                                  # full metadata on EVERY point
        "id", "field", "category", "embedded_text",
        "objective", "work_exp", "information",
        "certification", "technical_skills",
    },
}
```

Why regex **before** GPT-4.1: `Resume_html` already carries a canonical schema —
every section is tagged `id="SECTION_<CODE>…"` (`SUMM`, `EXPR`, `EDUC`, `SKLL`, `CERT`, …)
and jobs are tagged `jobtitle` / `companyname` / `jobdates` / `joblocation`.
Regex extracts a **draft** for free; GPT-4.1 then fixes mistakes, mines certifications
mentioned outside a CERT section, and normalizes skills into a clean list.

Outputs land in `output/`: `records.json` (structured resumes) and `points.jsonl` (embedding data points).

In [ ]:
!pip install tqdm

: 

In [ ]:
# --- setup & config -----------------------------------------------------------
import os, re, json, html, uuid, time, collections
import concurrent.futures as cf
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv

load_dotenv()                              # reads .env next to this notebook

DATA_CSV  = Path("data/Resume/Resume.csv")
PDF_DIR   = Path("data/data/data/INFORMATION-TECHNOLOGY")
OUT_DIR   = Path("output")
GPT_CACHE = OUT_DIR / "gpt_cache"          # 1 json per resume -> reruns are free
OUT_DIR.mkdir(exist_ok=True)
GPT_CACHE.mkdir(exist_ok=True)

CATEGORY    = "INFORMATION-TECHNOLOGY"
GPT_MODEL   = "gpt-4.1"
EMBED_MODEL = "text-embedding-3-small"
EMBED_DIMS  = 1536

QDRANT_URL     = os.environ.get("QDRANT_URL", "http://localhost:6333")
QDRANT_API_KEY = os.environ.get("QDRANT_API_KEY", "")   # storage/config/qdrant/.env
QDRANT_HEADERS = {"api-key": QDRANT_API_KEY} if QDRANT_API_KEY else {}

key = os.environ.get("OPENAI_API_KEY", "")
if not key or "REPLACE_ME" in key:
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")
print("OPENAI_API_KEY loaded:", os.environ["OPENAI_API_KEY"][:7] + "...")

In [ ]:
# --- load: CSV filtered to IT, cross-checked against the PDF folder ------------
df    = pd.read_csv(DATA_CSV)
df_it = df[df.Category == CATEGORY].reset_index(drop=True)

pdf_ids = {int(p.stem) for p in PDF_DIR.glob("*.pdf")}
csv_ids = set(df_it.ID.astype(int))
assert csv_ids == pdf_ids, f"CSV/PDF folder mismatch: {sorted(csv_ids ^ pdf_ids)}"

print(f"{len(df_it)} resumes | CSV IDs match {PDF_DIR.name}/ PDFs: {csv_ids == pdf_ids}")
df_it[["ID", "Category"]].head(3)

## Step 1 — regex extraction from `Resume_html`

Section codes found in this corpus and how they map to our fields:

| code | meaning | → field |
|---|---|---|
| `SUMM` `PRFL` | summary / profile | `objective` |
| `EXPR` (`WRKH` fallback) | experience / work history | `work_exp` |
| `CERT` | certifications | `certification` |
| `SKLL` `TSKL` `HILT` | skills / technical skills / highlights | `technical_skills` |
| `EDUC` `LANG` `ACCM` `AFIL` `ADDI` `INTR` `CUST` … | everything else | `information` |
| `NAME` `ALNK` | candidate name / personal links | **excluded** (PII) |

Robustness details (each one traced to real documents in this corpus):

- Some docs use UUID section ids (`SECTION_NAMEa29f1cb8-…`) — the regex relies only on the 4-letter code.
- Section headings are removed **structurally** (the `SECTNAME_…` div), so any wording works.
- In one template family the `to` between dates is itself a `jobdates` span — filtered out.
- Docs with both `EXPR` and `WRKH` duplicate every job — `WRKH` is only used when `EXPR` is empty, plus dedup.
- A PII scrub drops LinkedIn/GitHub URLs, emails, phones, and salary lines that survive the source anonymization.

In [ ]:
# --- regex helpers --------------------------------------------------------------
def clean(t: str) -> str:
    """Unescape entities, normalize unicode junk, collapse whitespace."""
    t = html.unescape(t)
    for a, b in [("\xa0", " "), ("​", ""), ("\xad", ""), ("－", "-")]:
        t = t.replace(a, b)
    return re.sub(r"\s+", " ", t).strip()

def strip_tags(seg: str) -> str:
    return clean(re.sub(r"<[^>]+>", " ", seg))

# personal identifiers that survive the corpus anonymization (links, contacts, salary)
PII_RES = [
    re.compile(r"(?:https?://)?(?:www\.)?(?:linkedin\.com/in/|github\.com/)\S+", re.I),
    re.compile(r"[\w.+-]+@[\w-]+\.\w+"),
    re.compile(r"\(?\d{3}\)?[-.\s]\d{3}[-.\s]\d{4}"),
    re.compile(r"(?:Starting|Ending)\s+Salary:\s*\$[\d,]+", re.I),
]
def scrub_pii(t: str) -> str:
    for rx in PII_RES:
        t = rx.sub(" ", t)
    return re.sub(r"[ \t]{2,}", " ", t).strip()   # tidy, but keep newlines

# every section div: id="SECTION_<4-letter code><number or uuid>"
SEC_RE = re.compile(r'<div[^>]*id="SECTION_([A-Z]{4})')
# the rendered heading lives in its own div (id="SECTNAME_...") -> strip structurally,
# so ANY heading wording ("Qualifications", "Certifications and Training", ...) is removed
SECTNAME_RE = re.compile(r'<div[^>]*id="SECTNAME_[^"]*"[^>]*>.*?</div>', re.S)

def split_sections(h: str) -> dict:
    """{4-letter code: [raw html segment, ...]} — slice between section starts."""
    ms  = list(SEC_RE.finditer(h))
    out = collections.defaultdict(list)
    for m, nxt in zip(ms, ms[1:] + [None]):
        out[m.group(1)].append(h[m.start(): nxt.start() if nxt else len(h)])
    return out

def grab(cls: str, seg: str) -> list:
    """Non-empty text of every element whose class starts with `cls`."""
    return [t for t in (clean(re.sub(r"<[^>]+>", " ", m)) for m in
            re.findall(r'<[^>]*class="' + cls + r'[^"]*"[^>]*>(.*?)</', seg, re.S)) if t]

PARA_RE = re.compile(r'<div class="paragraph[^"]*"[^>]*>')

def split_paragraphs(seg: str) -> list:
    ms = list(PARA_RE.finditer(seg))
    return [seg[m.start(): nxt.start() if nxt else len(seg)]
            for m, nxt in zip(ms, ms[1:] + [None])]

# strip the job header markup so only the description remains
DATE_JUNK = re.compile(r'<(span|div)[^>]*class="[^"]*(dates_?[Ww]rapper|jobdates)[^"]*"[^>]*>.*?</\1>', re.S)
HDR_JUNK  = re.compile(r'<span[^>]*class="(jobtitle|companyname|joblocation)[^"]*"[^>]*>.*?</span>', re.S)
# separator junk before AND after the optional 'to' token (incl. en/em dash, bullets)
DESC_JUNK = re.compile(r"^[\s,;:.\-–—|/•*]*(?:to\s+[\s,;:.\-–—|/•*]*)?")

def parse_jobs(seg: str) -> list:
    jobs = []
    for p in split_paragraphs(seg):
        title = grab("jobtitle", p)
        comp  = grab("companyname", p)
        # the ' to ' range separator is itself a jobdates span in one template family
        dates = [d for d in grab("jobdates", p) if d.lower() != "to"]
        loc   = grab("joblocation", p)
        body  = HDR_JUNK.sub(" ", DATE_JUNK.sub(" ", p))
        desc  = DESC_JUNK.sub("", strip_tags(body))
        if not (title or dates or desc):       # placeholder-only template rows
            continue
        jobs.append({
            "title":       title[0] if title else None,
            "company":     comp[0]  if comp  else None,
            "start_date":  dates[0] if dates else None,
            "end_date":    dates[1] if len(dates) > 1 else None,
            "location":    " ".join(loc) or None,
            "description": desc or None,
        })
    return jobs

FIELD_MAP = {
    "objective":        ["SUMM", "PRFL"],       # CUST is arbitrary custom content -> information
    "work_exp":         ["EXPR", "WRKH"],
    "certification":    ["CERT"],
    "technical_skills": ["SKLL", "TSKL", "HILT"],
}
# NAME (candidate name) and ALNK (personal links) are excluded entirely
EXCLUDE  = {c for v in FIELD_MAP.values() for c in v} | {"NAME", "ALNK"}
TITLE_RE = re.compile(
    r"^(Summary|Professional Summary|Career Overview|Executive Profile|Skills"
    r"|Skill Highlights|Highlights|Core Qualifications|Certifications"
    r"|Experience|Work History|Professional Experience)\s*", re.I)

def sec_text(s: str) -> str:
    """Section html -> text, with the heading div removed structurally."""
    return TITLE_RE.sub("", strip_tags(SECTNAME_RE.sub(" ", s)))

def extract(row) -> dict:
    """One resume row -> draft record via HTML structure only."""
    secs = split_sections(row.Resume_html)
    rec  = {"id": int(row.ID)}
    for field, codes in FIELD_MAP.items():
        if field == "work_exp":
            jobs, seen = [], set()
            for c in codes:
                for s in secs.get(c, []):
                    for j in parse_jobs(s):
                        k = (j["title"], j["company"], j["start_date"], j["end_date"], j["description"])
                        if k not in seen:
                            seen.add(k)
                            jobs.append(j)
                if jobs:
                    break                       # WRKH always duplicates EXPR in this corpus
            rec[field] = jobs
        else:
            texts = [sec_text(s) for c in codes for s in secs.get(c, [])]
            rec[field] = scrub_pii(" | ".join(t for t in texts if t)) or None
    info = [sec_text(s) for c, ss in secs.items() if c not in EXCLUDE for s in ss]
    rec["information"] = scrub_pii(" | ".join(t for t in info if t)) or None
    return rec

In [ ]:
# --- run the draft extraction on all resumes ------------------------------------
draft_records = [extract(r) for r in df_it.itertuples()]

fields = ["objective", "work_exp", "certification", "technical_skills", "information"]
cov    = {f: sum(1 for r in draft_records if r.get(f)) for f in fields}
print(f"coverage / {len(draft_records)} docs:", cov)

n_jobs = sorted(len(r["work_exp"]) for r in draft_records)
print(f"jobs per doc: min {n_jobs[0]}, median {n_jobs[len(n_jobs)//2]}, max {n_jobs[-1]}")

print("\n--- sample draft ---")
s = draft_records[0]
print(json.dumps({**s, "work_exp": s["work_exp"][:1],
                  "information": (s["information"] or "")[:150]},
                 indent=2, ensure_ascii=False)[:1500])

## Step 2 — GPT-4.1 normalization

GPT-4.1 receives the **draft** (regex) + the **full plain text** and returns the final
record under a strict JSON schema. It:

- cleans `objective` (or returns `null` if the resume truly has none — no invention),
- fixes job entries the regex mis-split,
- mines `certification` mentioned *anywhere* (regex found a CERT section in only ~20/120 docs),
- normalizes `technical_skills` into a deduplicated list of short noun phrases,
- structures `information` into education / languages / other.

Results are cached to `output/gpt_cache/<id>.json`, so re-running is free.

In [ ]:
# --- GPT-4.1: strict-schema extraction ------------------------------------------
import openai
from openai import OpenAI
client = OpenAI()

# only transient errors are worth retrying; 401/400/refusals should fail fast
RETRYABLE = (openai.RateLimitError, openai.APIConnectionError,
             openai.APITimeoutError, openai.InternalServerError)

JOB = {"type": "object", "additionalProperties": False,
       "properties": {"title":       {"type": ["string", "null"]},
                      "company":     {"type": ["string", "null"]},
                      "start_date":  {"type": ["string", "null"]},
                      "end_date":    {"type": ["string", "null"]},
                      "location":    {"type": ["string", "null"]},
                      "description": {"type": ["string", "null"]}},
       "required": ["title", "company", "start_date", "end_date", "location", "description"]}

EDU = {"type": "object", "additionalProperties": False,
       "properties": {"degree":      {"type": ["string", "null"]},
                      "field":       {"type": ["string", "null"]},
                      "institution": {"type": ["string", "null"]},
                      "year":        {"type": ["string", "null"]}},
       "required": ["degree", "field", "institution", "year"]}

SCHEMA = {"type": "object", "additionalProperties": False,
          "properties": {
              "objective":   {"type": ["string", "null"]},
              "work_exp":    {"type": "array", "items": JOB},
              "information": {"type": "object", "additionalProperties": False,
                              "properties": {"education": {"type": "array", "items": EDU},
                                             "languages": {"type": "array", "items": {"type": "string"}},
                                             "other":     {"type": ["string", "null"]}},
                              "required": ["education", "languages", "other"]},
              "certification":    {"type": "array", "items": {"type": "string"}},
              "technical_skills": {"type": "array", "items": {"type": "string"}}},
          "required": ["objective", "work_exp", "information", "certification", "technical_skills"]}

SYSTEM_PROMPT = """You are a resume-parsing engine. You receive the full plain text of one \
resume plus a DRAFT extraction produced by regex over the resume's HTML. Return the final \
extraction as JSON following the schema exactly.

Rules:
- Prefer the draft's structure, but fix its mistakes and fill gaps by reading the full text.
- objective: the candidate's summary/objective as one clean paragraph; null if the resume has none. Never invent one.
- work_exp: one entry per job, most recent first, no duplicate entries. Keep descriptions essentially as written (light cleanup only). Keep dates as they appear (e.g. "06/2015", "Jan 2014", "Current").
- certification: certifications/licenses found ANYWHERE in the resume, one string each ("name, issuer/year" when given). Empty list if none.
- technical_skills: deduplicated list of concrete skills, tools and technologies from the skills/highlights sections AND job descriptions. Short noun phrases.
- information: education entries, languages, and any other notable content (affiliations, accomplishments, awards, additional info) summarized in "other".
- Omit personal identifiers everywhere: candidate names, emails, phone numbers, personal URLs (LinkedIn/GitHub/websites), street addresses, and salary figures.
- Never fabricate content. "Company Name" and "City, State" are anonymization placeholders - keep them verbatim where they appear."""

def gpt_extract(draft: dict, full_text: str) -> dict:
    cache = GPT_CACHE / f"{draft['id']}.json"
    if cache.exists():
        try:
            return json.loads(cache.read_text())
        except (json.JSONDecodeError, OSError):
            cache.unlink(missing_ok=True)      # corrupt cache (interrupted write) -> redo
    # restore line structure: the flat text encodes line breaks as runs of 2+ spaces
    plain = scrub_pii(re.sub(r"\s{2,}", "\n", full_text).strip())
    user  = (f"DRAFT (regex extraction):\n{json.dumps(draft, ensure_ascii=False)}\n\n"
             f"FULL RESUME TEXT:\n{plain}")
    last_err = None
    for attempt in range(5):
        try:
            resp = client.chat.completions.create(
                model=GPT_MODEL, temperature=0,
                response_format={"type": "json_schema",
                                 "json_schema": {"name": "resume_extraction",
                                                 "strict": True, "schema": SCHEMA}},
                messages=[{"role": "system", "content": SYSTEM_PROMPT},
                          {"role": "user",   "content": user}])
            choice = resp.choices[0]
            if getattr(choice.message, "refusal", None):       # don't retry, surface it
                raise RuntimeError(f"model refused id={draft['id']}: {choice.message.refusal}")
            if choice.finish_reason == "length":               # identical retry would re-truncate
                raise RuntimeError(f"output truncated (finish_reason=length) id={draft['id']}")
            out = json.loads(choice.message.content)
            out["id"] = draft["id"]
            tmp = cache.with_suffix(".json.tmp")               # atomic write:
            tmp.write_text(json.dumps(out, ensure_ascii=False))
            os.replace(tmp, cache)                             # cache is absent or complete
            return out
        except RETRYABLE as e:
            last_err = e
            if attempt < 4:                    # TPM 429s need 10-15s to clear (30k TPM tier)
                time.sleep(min(30, 2 * 2 ** attempt))
    raise RuntimeError(f"GPT extraction failed for id={draft['id']}: {last_err!r}") from last_err

In [ ]:
# --- run GPT-4.1 over all resumes (concurrent, cached) --------------------------
raw_by_id = dict(zip(df_it.ID.astype(int), df_it.Resume_str))

# 3 workers stays under a 30k tokens/min org limit (8 workers triggered 429 storms)
records, errors = [], []
with cf.ThreadPoolExecutor(max_workers=3) as ex:
    futs = {ex.submit(gpt_extract, d, raw_by_id[d["id"]]): d["id"] for d in draft_records}
    for fut in tqdm(cf.as_completed(futs), total=len(futs), desc="gpt-4.1"):
        try:
            records.append(fut.result())
        except Exception as e:
            errors.append((futs[fut], repr(e)))

records.sort(key=lambda r: r["id"])
print(f"ok: {len(records)}  failed: {len(errors)}")

# don't ship a partial corpus: successes are cached, so a re-run only retries failures
if errors:
    for rid, err in errors[:10]:
        print(f"  {rid}: {err}")
    raise RuntimeError(f"{len(errors)} resumes failed - re-run this cell (cached ids are free)")

(OUT_DIR / "records.json").write_text(
    json.dumps(records, ensure_ascii=False, indent=1))
print("saved ->", OUT_DIR / "records.json")

## Step 3 — embeddings → data points

One vector **per non-empty field** per resume (≈5 × 120 ≈ 550–600 points).
Every point carries the **full metadata** of its resume in the payload, plus
`field` (which text this vector encodes) and `embedded_text` (the exact input string).

Point ids are `uuid5(resume_id + field)` — deterministic, and valid Qdrant point ids
(Qdrant only accepts unsigned ints or UUIDs, not arbitrary strings).

In [ ]:
# --- render each field to text, embed, build points ------------------------------
import tiktoken
ENC     = tiktoken.get_encoding("cl100k_base")   # tokenizer used by text-embedding-3-*
MAX_TOK = 8000                                   # headroom under the 8192-token input cap

EMBED_FIELDS = ["objective", "work_exp", "information", "certification", "technical_skills"]

def render_job(j: dict) -> str:
    head = " | ".join(str(x) for x in [
        j.get("title"), j.get("company"),
        f"{j.get('start_date') or '?'} - {j.get('end_date') or '?'}",
        j.get("location")] if x)
    return f"{head}: {j.get('description') or ''}".strip(" :")

def render_field(rec: dict, field: str):
    v = rec.get(field)
    if not v:
        return None
    if field == "work_exp":
        return "\n".join(render_job(j) for j in v) or None
    if field == "information" and isinstance(v, dict):   # GPT shape (drafts keep a string)
        edu = "; ".join(" ".join(str(p) for p in [e.get("degree"), e.get("field"),
                                                  e.get("institution"), e.get("year")] if p)
                        for e in v.get("education", []))
        parts = [p for p in [edu, ", ".join(v.get("languages", [])), v.get("other")] if p]
        return "\n".join(parts) or None
    if isinstance(v, list):
        return "; ".join(v) or None
    return str(v).strip() or None

# (point_id, record, field, text, n_tokens) — truncated ONCE, so the payload's
# embedded_text is by construction the exact string the vector encodes
items = []
for rec in records:
    for field in EMBED_FIELDS:
        text = render_field(rec, field)
        if not text:
            continue
        toks = ENC.encode(text)
        if len(toks) > MAX_TOK:
            toks = toks[:MAX_TOK]
            text = ENC.decode(toks)
        pid = str(uuid.uuid5(uuid.NAMESPACE_URL, f"cv:{rec['id']}:{field}"))
        items.append((pid, rec, field, text, len(toks)))
print(f"{len(items)} texts to embed "
      f"({collections.Counter(f for _, _, f, _, _ in items)})")

def embed_batch(texts: list, attempts: int = 4) -> list:
    last_err = None
    for a in range(attempts):
        try:
            resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
            return [d.embedding for d in sorted(resp.data, key=lambda d: d.index)]
        except RETRYABLE as e:
            last_err = e
            if a < attempts - 1:
                time.sleep(2 ** a)
    raise RuntimeError(f"embedding batch failed: {last_err!r}") from last_err

# batch by count AND token budget (API caps: 2048 inputs / 300k tokens per request)
batches, cur, cur_tok = [], [], 0
for *_, text, n in items:
    if cur and (len(cur) == 64 or cur_tok + n > 250_000):
        batches.append(cur)
        cur, cur_tok = [], 0
    cur.append(text)
    cur_tok += n
if cur:
    batches.append(cur)

vectors = []
for b in tqdm(batches, desc="embeddings"):
    vectors.extend(embed_batch(b))
assert len(vectors) == len(items), f"{len(vectors)} vectors for {len(items)} items"

points = [{"id": pid,
           "vector": vec,
           "payload": {"id": rec["id"], "field": field, "category": CATEGORY,
                       "embedded_text": text,
                       **{f: rec[f] for f in EMBED_FIELDS}}}
          for (pid, rec, field, text, _), vec in zip(items, vectors, strict=True)]
print(f"{len(points)} data points, {len(points[0]['vector'])} dims")

In [ ]:
# --- save + preview one data point ----------------------------------------------
with open(OUT_DIR / "points.jsonl", "w") as fh:
    for p in points:
        fh.write(json.dumps(p, ensure_ascii=False) + "\n")
print("saved ->", OUT_DIR / "points.jsonl")

p = next(pt for pt in points if pt["payload"]["field"] == "objective")
preview = {"id": p["id"],
           "vector": [round(x, 4) for x in p["vector"][:6]] + ["..."],
           "payload": {k: (v[:120] + "..." if isinstance(v, str) and len(v) > 120 else
                           v[:1] if isinstance(v, list) and k == "work_exp" else v)
                       for k, v in p["payload"].items()}}
print(json.dumps(preview, indent=2, ensure_ascii=False)[:2000])

In [ ]:
# --- upsert into Qdrant (storage stack; url + api-key from .env) -----------------
# Idempotent: point ids are deterministic uuid5, so re-running just overwrites.
UPSERT_TO_QDRANT = True

COLLECTION = "cv_information_technology"

if UPSERT_TO_QDRANT:
    import requests
    r = requests.put(f"{QDRANT_URL}/collections/{COLLECTION}", headers=QDRANT_HEADERS,
                     json={"vectors": {"size": EMBED_DIMS, "distance": "Cosine"}})
    print("create collection:", r.status_code, r.text[:100])
    if r.status_code == 409:      # already exists (rerun) -> config must match
        v = requests.get(f"{QDRANT_URL}/collections/{COLLECTION}",
                         headers=QDRANT_HEADERS).json()["result"]["config"]["params"]["vectors"]
        assert v["size"] == EMBED_DIMS and v["distance"] == "Cosine", (
            f"collection {COLLECTION!r} exists with incompatible config {v} - "
            f"delete it (DELETE /collections/{COLLECTION}) or change COLLECTION")
    elif r.status_code != 200:
        raise RuntimeError(f"create collection failed ({r.status_code}): {r.text[:300]}")

    # payload indexes for the filter keys used in per-field / per-resume search
    for name, schema in [("field", "keyword"), ("id", "integer"), ("category", "keyword")]:
        requests.put(f"{QDRANT_URL}/collections/{COLLECTION}/index?wait=true",
                     headers=QDRANT_HEADERS,
                     json={"field_name": name, "field_schema": schema})

    for i in tqdm(range(0, len(points), 100), desc="qdrant upsert"):
        r = requests.put(f"{QDRANT_URL}/collections/{COLLECTION}/points?wait=true",
                         headers=QDRANT_HEADERS, json={"points": points[i:i + 100]})
        if not r.ok:              # surface Qdrant's error body, not just the status line
            raise RuntimeError(f"upsert failed ({r.status_code}): {r.text[:300]}")

    info = requests.get(f"{QDRANT_URL}/collections/{COLLECTION}", headers=QDRANT_HEADERS).json()
    print("points in collection:", info["result"]["points_count"])